# 第二次作业 07 降维实验与地理对齐

核验降维输入，比较 MDS、t-SNE、UMAP，再检查参数、种子和地理对齐。

原编号与新文件名的对应关系、外部依赖和执行说明见 [第二次作业导航](hw02_README.md)。

先运行本文件的**环境初始化**单元格；重启内核后也应先运行初始化。各节按原顺序排列，依赖上一步生成的变量时会在节前列出。

已有输出和执行编号是整理前的运行记录，未重新生成；新增初始化单元格没有历史输出。

UMAP 安装步骤仅在缺少依赖时手动使用；已有依赖无需再安装。两组完全相同的早期基线代码已合并，保留最后成功运行的基线与核验结果；原始失败输出完整保存在整理前备份中。

[上一份 06](hw02_06_time_matrix.ipynb) · [下一份 08](hw02_08_clustering.ipynb)

## 环境初始化

此单元格集中加载公共依赖并检查脚本位置；不执行采集、安装或模型拟合。

In [ ]:
# 标准库
import importlib
import importlib.metadata as metadata
import sys
from pathlib import Path

# 第三方库
import pandas as pd
from IPython.display import display

# 仅在这里配置本机项目位置。脚本和来源数据仍保留在原 railway_analysis 项目中。
ROOT = Path(r"D:\yangz\大三上\机器学习\第二次作业\railway_analysis")
EXPECTED_PYTHON = Path(r"D:\anaconda\envs\pda312\python.exe")
SCRIPTS_DIR = ROOT / "scripts"

if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError("请选择 pda312 内核，再从本文件的环境初始化单元格开始运行。")
if not SCRIPTS_DIR.is_dir():
    raise FileNotFoundError(f"未找到作业脚本目录：{SCRIPTS_DIR}。请核对上方 ROOT 配置。")

REQUIRED_LOCAL_MODULES = (
    "stage2_baseline",
    "stage2_environment",
    "stage2_experiments",
    "stage2_geography",
    "stage2_inputs",
)
missing_modules = [
    name for name in REQUIRED_LOCAL_MODULES
    if not (SCRIPTS_DIR / f"{name}.py").is_file()
]
if missing_modules:
    raise FileNotFoundError(f"作业脚本缺失：{', '.join(missing_modules)}")

# 重复运行初始化时不重复插入同一路径；业务模块仍在对应步骤导入和 reload。
scripts_path = str(SCRIPTS_DIR)
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)


## 问题2—辅助代码0：UMAP 依赖检查与按需安装（可选）

In [1]:
# 功能：已有UMAP则报告版本；没有则安装umap-learn及其必要依赖。
# 约束：保持当前numpy、scipy、scikit-learn、pandas、matplotlib版本；不会删除原文件。
# 输入：当前pda312环境；需要网络。通常1—5分钟，下载速度会影响耗时。
# 输出：environment/user_umap_install_*/安装命令、版本约束、pip真实日志、安装结果。
# 安装失败时不去掉约束反复强装；回传末尾错误即可。
import stage2_environment
importlib.reload(stage2_environment)
install_status = stage2_environment.install_umap_user()
print(install_status)
print("新安装完成后，请重启pda312内核，再从核心1开始；不要在旧内核直接继续。")


正在安装UMAP及其依赖；现有numpy/scipy/sklearn/pandas/matplotlib版本已约束。
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple/
     ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
     ----------- ---------------------------- 0.8/2.8 MB 5.6 MB/s eta 0:00:01
     ------------------------- -------------- 1.8/2.8 MB 4.6 MB/s eta 0:00:01
     ---------------------------------------- 2.8/2.8 MB 5.3 MB/s  0:00:00
     ---------------------------------------- 0.0/41.9 MB ? eta -:--:--
     -- ------------------------------------- 2.4/41.9 MB 11.2 MB/s eta 0:00:04
     -- ------------------------------------- 2.9/41.9 MB 8.4 MB/s eta 0:00:05
     --- ------------------------------------ 3.1/41.9 MB 5.3 MB/s eta 0:00:08
     --- ------------------------------------ 3.4/41.9 MB 3.9 MB/s eta 0:00:10
     --- ------------------------------------ 3.9/41.9 MB 3.7 MB/s eta 0:00:11
     ---- ----------------------------------- 4.2/41.9 MB 3.4 MB/s eta 0:00:11
     ---- --------------------

## 问题2—核心代码1：核验最终数据并冻结降维输入

In [1]:
# 功能：离线回放来源，检查9份34节点矩阵，保存13份后续分析输入及哈希。
import stage2_inputs
importlib.reload(stage2_inputs)
versions = {p: metadata.version(p) for p in [
    "numpy", "scipy", "scikit-learn", "pandas", "umap-learn",
]}
display(pd.Series(versions, name="version").to_frame())
inputs = stage2_inputs.prepare(actor="user")
INPUT_OUT = Path(inputs["output_dir"])
display(pd.read_csv(INPUT_OUT / "final_quality_summary.csv", encoding="utf-8-sig"))
inventory = pd.read_csv(INPUT_OUT / "matrix_inventory.csv", encoding="utf-8-sig")
display(inventory.loc[inventory["baseline"].eq(True), [
    "dataset_id", "node_count", "access_min", "alpha", "role",
]])
print("冻结矩阵数：", inputs["matrix_count"], "；统一缩放分钟：", inputs["common_scale_minutes"])
print("最终来源/明确推导区间记录数：", inputs["source_record_count"], "（不是独立列车数）")
print("输入目录：", INPUT_OUT)


,version
numpy,2.5.0
scipy,1.18.0
scikit-learn,1.9.0
pandas,3.0.3
umap-learn,0.6.0


,check,value
0,final_source_and_derived_rows,28323.0
1,ordinary_missing_in_raw34,0.0
2,finite_directed_pairs_33,1056.0
3,Taipei_structural_directed_cells,66.0
4,ready_full34_variants,9.0
5,source_conflicts,0.0
6,common_positive_distance_scale_minutes,537.5


,dataset_id,node_count,access_min,alpha,role
4,access_60_alpha_5_n34,34,60.0,5.0,full34_with_structural_penalty
10,access_60_n33,33,60.0,NaN,no_Taipei_control_with_explicit_Macao_access


冻结矩阵数： 13 ；统一缩放分钟： 537.5
最终来源/明确推导区间记录数： 28323 （不是独立列车数）
输入目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\analysis\user_stage2_inputs_20261114_143755_625472


## 问题2—核心代码2：运行MDS、t-SNE、UMAP六次基线（保留的成功运行版本）

In [4]:
# 功能：34节点主情景和去台北33节点对照，各运行三算法，保存二维坐标、参数、质量指标和耗时。
# 输入：核心1冻结的两份基线矩阵；纯本地分析，不采集新数据。
# MDS：度量MDS，2维，n_init=4，max_iter=1000，eps=1e-6，随机初始化。
# t-SNE：perplexity=10，max_iter=1000，learning_rate=auto，early_exaggeration=12，exact，随机初始化。
# UMAP：n_neighbors=10，min_dist=0.1，n_epochs=500，随机初始化。
# 三算法均metric/precomputed，seed=42，单工作线程；适配当前scikit-learn的MDS接口。
import stage2_inputs
import stage2_baseline
importlib.reload(stage2_inputs)
importlib.reload(stage2_baseline)
baseline = stage2_baseline.run(actor="user")
for key in [
    "completed_runs", "expected_runs", "baseline_seed",
    "parameter_experiments_complete", "three_seed_stability_complete", "output_dir",
]:
    print(f"{key}: {baseline[key]}")
print("本批基线完成后继续核心3；参数实验和多种子稳定性尚未执行。")


正在运行： access_60_alpha_5_n34_MDS_seed42
完成： MDS ；外部stress= 0.0364 ；秩相关= 0.9717
正在运行： access_60_alpha_5_n34_tSNE_seed42
完成： tSNE ；外部stress= 0.5466 ；秩相关= 0.7164
正在运行： access_60_alpha_5_n34_UMAP_seed42
完成： UMAP ；外部stress= 0.927 ；秩相关= 0.5811
正在运行： access_60_n33_MDS_seed42
完成： MDS ；外部stress= 0.1792 ；秩相关= 0.967
正在运行： access_60_n33_tSNE_seed42
完成： tSNE ；外部stress= 0.547 ；秩相关= 0.7662
正在运行： access_60_n33_UMAP_seed42
完成： UMAP ；外部stress= 0.6384 ；秩相关= 0.6219
completed_runs: 6
expected_runs: 6
baseline_seed: 42
parameter_experiments_complete: False
three_seed_stability_complete: False
output_dir: D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\analysis\user_stage2_baseline_20261114_145338_648190
本批基线完成后继续核心3；参数实验和多种子稳定性尚未执行。


## 问题2—核心代码3：复核六次坐标及统一质量指标（保留的成功运行版本）

In [5]:
# 功能：从磁盘复核输入/代码/坐标哈希、节点顺序和实际坐标重算指标；显示全部及连通网络内部的质量。
# 输入：核心2的已保存真实运行结果；通常数秒。新内核先运行环境初始化，再运行本段。
# 输出：同一结果目录内baseline_verification.json；不重新运行降维。
# 外部stress：拟合一个统一坐标尺度后，相对原距离的残差，越小越好；用于跨算法一致比较。
# distance_spearman：原距离与二维距离的秩相关，越大越好。
# neighbor_overlap_k5、trustworthiness_k5：局部5近邻保留与可信度，越大越好。
# network_only指标：34节点结果只取其中33个连通节点的坐标评估，未重新拟合；33节点对照则评估全部33点。
# 台北等惩罚产生并列距离；近邻重叠用固定节点顺序处理并列，须结合33节点对照解释。
# MDS自身stress、t-SNE自身KL的定义不同，不相互比较；UMAP没有同定义的公开最终损失。
# 不根据坐标轴数值解释经纬度或分钟；这里没有捏造迭代收敛曲线。
import stage2_inputs
import stage2_baseline
importlib.reload(stage2_inputs)
importlib.reload(stage2_baseline)
verified = stage2_baseline.verify_saved()
BASELINE_OUT = Path(verified["output_dir"])
metrics = pd.read_csv(BASELINE_OUT / "baseline_metrics.csv", encoding="utf-8-sig")
with pd.option_context("display.max_columns", None, "display.width", 200, "display.precision", 5):
    display(metrics[[
        "dataset_id", "algorithm", "node_count", "seconds",
        "external_scale_fitted_stress1", "distance_spearman", "neighbor_overlap_k5",
        "trustworthiness_k5",
    ]])
    display(metrics[[
        "dataset_id", "algorithm", "network_only_external_scale_fitted_stress1",
        "network_only_distance_spearman", "network_only_neighbor_overlap_k5",
        "network_only_trustworthiness_k5",
    ]])
print("六次保存结果核验：", verified)


,dataset_id,algorithm,node_count,seconds,external_scale_fitted_stress1,distance_spearman,neighbor_overlap_k5,trustworthiness_k5
0,access_60_alpha_5_n34,MDS,34,0.05494,0.03641,0.97167,0.75294,0.95656
1,access_60_alpha_5_n34,tSNE,34,0.20845,0.54661,0.71638,0.67059,0.87104
2,access_60_alpha_5_n34,UMAP,34,0.02872,0.92699,0.58109,0.71765,0.87081
3,access_60_n33,MDS,33,0.07581,0.17919,0.96699,0.74545,0.95636
4,access_60_n33,tSNE,33,0.19695,0.54700,0.76617,0.74545,0.90885
5,access_60_n33,UMAP,33,0.02520,0.63835,0.62191,0.73939,0.87321


,dataset_id,algorithm,network_only_external_scale_fitted_stress1,network_only_distance_spearman,network_only_neighbor_overlap_k5,network_only_trustworthiness_k5
0,access_60_alpha_5_n34,MDS,0.18079,0.96614,0.76970,0.96000
1,access_60_alpha_5_n34,tSNE,0.60422,0.65996,0.69091,0.88679
2,access_60_alpha_5_n34,UMAP,0.63116,0.63530,0.73333,0.88582
3,access_60_n33,MDS,0.17919,0.96699,0.74545,0.95636
4,access_60_n33,tSNE,0.54700,0.76617,0.74545,0.90885
5,access_60_n33,UMAP,0.63835,0.62191,0.73939,0.87321


六次保存结果核验： {'output_dir': 'D:\\yangz\\大三上\\机器学习\\第二次作业\\railway_analysis\\data\\analysis\\user_stage2_baseline_20261114_145338_648190', 'verified_runs': 6, 'coordinate_hashes_passed': True, 'node_order_passed': True, 'external_metrics_reproduced': True, 'parameter_experiments_complete': False, 'three_seed_stability_complete': False}


## 问题2—核心代码4：核验基线并冻结参数实验表

In [6]:
# 功能：检查本轮6个基线，生成82行去重实验表、来源协议及计划哈希。
import stage2_experiments
importlib.reload(stage2_experiments)
plan = stage2_experiments.preview(actor="user")
for key in ["total_unique_runs", "reusable_baseline_runs", "new_model_fits", "seeds", "output_dir"]:
    print(f"{key}: {plan[key]}")
display(pd.Series(plan["tag_counts"], name="comparison_rows").rename_axis("comparison").to_frame())
plan_table = pd.read_csv(Path(plan["output_dir"]) / "experiment_plan.csv", encoding="utf-8-sig")
display(plan_table.groupby(["model", "node_count"]).size().rename("unique_runs").to_frame())
assert len(plan_table) == 82 and plan_table["run_id"].nunique() == 82


total_unique_runs: 82
reusable_baseline_runs: 6
new_model_fits: 76
seeds: [0, 42, 2026]
output_dir: D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\analysis\user_stage2_experiment_plan_20261114_151313_945440


,comparison_rows
comparison,
MDS_metric,4
MDS_n_init,6
UMAP_min_dist,6
UMAP_neighbors,6
UMAP_selected_combinations,4
representative_seeds,24
special_region_sensitivity,36
tSNE_iterations,6
tSNE_perplexity,6


unique_runs
model         node_count             
MDS           33                    7
              34                   13
MDS_nonmetric 33                    3
              34                    3
UMAP          33                   11
              34                   17
tSNE          33                   11
              34                   17

## 问题2—核心代码5：运行参数、种子和情景实验

In [7]:
# 功能：复用6个已核验基线，新增76次用户实际拟合；逐次保存坐标、参数、指标、警告及耗时。
import stage2_experiments
importlib.reload(stage2_experiments)
experiments = stage2_experiments.run(actor="user")
for key in [
    "completed_runs", "expected_runs", "reused_baseline_runs", "new_fits",
    "parameter_runs_complete", "verification_and_stability_summary_complete",
    "output_dir",
]:
    print(f"{key}: {experiments[key]}")


1/82 完成：MDS；stress=0.0364；rank=0.9717
2/82 正在拟合：access_60_alpha_5_n34_MDS_09bf7004153d
2/82 完成：MDS_nonmetric；stress=0.1765；rank=0.7767
3/82 正在拟合：access_60_alpha_5_n34_MDS_fd35c8a0c498
3/82 完成：MDS；stress=0.0369；rank=0.9725
4/82 正在拟合：access_60_alpha_5_n34_MDS_99a961db6edd
4/82 完成：MDS；stress=0.0364；rank=0.9718
5/82 正在拟合：access_60_alpha_5_n34_tSNE_8ae5b74783a1
5/82 完成：tSNE；stress=0.5093；rank=0.6461
6/82 完成：tSNE；stress=0.5466；rank=0.7164
7/82 正在拟合：access_60_alpha_5_n34_tSNE_2f18e135ce1e
7/82 完成：tSNE；stress=0.1043；rank=0.8879
8/82 正在拟合：access_60_alpha_5_n34_tSNE_e8561958ca16
8/82 完成：tSNE；stress=0.7629；rank=0.7056
9/82 正在拟合：access_60_alpha_5_n34_tSNE_d4f77ee1ce25
9/82 完成：tSNE；stress=0.2728；rank=0.7163
10/82 正在拟合：access_60_alpha_5_n34_tSNE_b90cf28c5aa8
10/82 完成：tSNE；stress=0.7010；rank=0.5792
11/82 正在拟合：access_60_alpha_5_n34_tSNE_f9b0fd96a187
11/82 完成：tSNE；stress=0.1201；rank=0.8879
12/82 正在拟合：access_60_alpha_5_n34_UMAP_68db5e750e3b
12/82 完成：UMAP；stress=0.9403；rank=0.3450
13/82 完成：UMAP；stress=0.

## 问题2—核心代码6：验收全部结果并汇总稳定性

In [8]:
# 功能：重读全部坐标，检查哈希、节点顺序、参数及外部指标；生成单参数、种子和特殊地区对照表。
import stage2_experiments
importlib.reload(stage2_experiments)
verified = stage2_experiments.verify_and_summarize()
EXPERIMENT_OUT = Path(verified["output_dir"])
quality = pd.read_csv(EXPERIMENT_OUT / "representative_seed_quality.csv", encoding="utf-8-sig")
stability = pd.read_csv(EXPERIMENT_OUT / "seed_stability_pairs.csv", encoding="utf-8-sig")
all_metrics = pd.read_csv(EXPERIMENT_OUT / "experiment_metrics.csv", encoding="utf-8-sig")
with pd.option_context("display.max_columns", None, "display.width", 200, "display.precision", 5):
    display(quality[[
        "dataset_id", "model", "seed_count",
        "network_only_external_scale_fitted_stress1_mean",
        "network_only_external_scale_fitted_stress1_sd",
        "network_only_neighbor_overlap_k5_mean",
    ]])
    display(stability.groupby(["dataset_id", "model"]).agg(
        seed_pairs=("seed_a", "size"),
        mean_disparity=("procrustes_disparity", "mean"),
        network_max_disparity=("network_only_procrustes_disparity", "max"),
        network_mean_neighbor_overlap=("network_only_embedding_neighbor_overlap_k5", "mean")))
    target_ids = [
        "access_60_alpha_2_n34", "access_60_alpha_5_n34", "access_60_alpha_10_n34",
        "access_60_n33",
    ]
    scenario = all_metrics.loc[all_metrics["tags"].str.contains("special_region_sensitivity") & all_metrics["dataset_id"].isin(target_ids)]
    display(scenario[[
        "dataset_id", "algorithm", "external_scale_fitted_stress1",
        "network_only_external_scale_fitted_stress1", "network_only_distance_spearman",
        "network_only_neighbor_overlap_k5",
    ]])
print("本批结果核验：", verified)


,dataset_id,model,seed_count,network_only_external_scale_fitted_stress1_mean,network_only_external_scale_fitted_stress1_sd,network_only_neighbor_overlap_k5_mean
0,access_60_alpha_5_n34,MDS,3,0.17748,0.00294,0.75556
1,access_60_alpha_5_n34,MDS_nonmetric,3,0.35151,0.07687,0.57980
2,access_60_alpha_5_n34,tSNE,3,0.58055,0.02732,0.70505
3,access_60_alpha_5_n34,UMAP,3,0.63719,0.00649,0.72323
4,access_60_n33,MDS,3,0.17499,0.00364,0.75758
5,access_60_n33,MDS_nonmetric,3,0.24865,0.00083,0.78384
6,access_60_n33,tSNE,3,0.58183,0.03016,0.74747
7,access_60_n33,UMAP,3,0.64437,0.01952,0.73333


seed_pairs  mean_disparity  network_max_disparity  network_mean_neighbor_overlap
dataset_id            model                                                                                          
access_60_alpha_5_n34 MDS                     3         0.01880                0.25847                        0.92525
                      MDS_nonmetric           3         0.00021                0.96933                        0.45859
                      UMAP                    3         0.04574                0.04801                        0.86667
                      tSNE                    3         0.29974                0.33477                        0.74949
access_60_n33         MDS                     3         0.21411                0.32779                        0.94747
                      MDS_nonmetric           3         0.18033                0.27079                        0.95556
                      UMAP                    3         0.35170                0.52121                        0.81818
                      tSNE                    3         0.25779                0.35578                        0.81818

,dataset_id,algorithm,external_scale_fitted_stress1,network_only_external_scale_fitted_stress1,network_only_distance_spearman,network_only_neighbor_overlap_k5
0,access_60_alpha_5_n34,MDS,0.03641,0.18079,0.96614,0.76970
5,access_60_alpha_5_n34,tSNE,0.54661,0.60422,0.65996,0.69091
12,access_60_alpha_5_n34,UMAP,0.92699,0.63116,0.63530,0.73333
26,access_60_n33,MDS,0.17919,0.17919,0.96699,0.74545
31,access_60_n33,tSNE,0.54700,0.54700,0.76617,0.74545
38,access_60_n33,UMAP,0.63835,0.63835,0.62191,0.73939
61,access_60_alpha_2_n34,MDS,0.08187,0.17609,0.96638,0.75152
62,access_60_alpha_2_n34,tSNE,0.42708,0.60422,0.65996,0.69091
63,access_60_alpha_2_n34,UMAP,0.84450,0.63116,0.63530,0.73333
64,access_60_alpha_10_n34,MDS,0.01848,0.18066,0.96594,0.76970


本批结果核验： {'output_dir': 'D:\\yangz\\大三上\\机器学习\\第二次作业\\railway_analysis\\data\\analysis\\user_stage2_experiments_20261114_151354_734440', 'verified_runs': 82, 'coordinate_hashes_passed': True, 'node_order_passed': True, 'external_metrics_reproduced': True, 'three_seeds': [0, 42, 2026], 'seed_stability_pairs': 24, 'scenario_comparisons': 36, 'parameter_comparison_rows': 42, 'parameter_runs_complete': True, 'representative_three_seed_stability_complete': True, 'geographic_comparison_complete': False, 'clustering_complete': False, 'note': 'Seed stability uses translation/rotation/reflection/scale-adjusted Procrustes disparity plus distance rank and neighbor overlap; not geographical alignment. No causal or final ranking inference from a single score.'}


## 问题2—核心代码7：核验34个城市参考点并建立地理距离

In [9]:
# 功能：核验来源字节/版本及34个唯一地理点，保存完整节点表、球面距离矩阵和投影经纬网。
import stage2_geography
importlib.reload(stage2_geography)
reference = stage2_geography.prepare_reference(actor="user")
REFERENCE_OUT = Path(reference["output_dir"])
print("城市参考点数：", reference["node_count"], "；坐标系：", reference["geography_crs"])
print("来源提交日期：", reference["source"]["source_commit_date"], "；固定版本：", reference["source"]["git_commit"])
print("距离方法：", reference["pairwise_method"])
print("投影口径：", reference["projection"])
geo_nodes = pd.read_csv(REFERENCE_OUT / "nodes_34_geographic_reference.csv", encoding="utf-8-sig")
display(geo_nodes.loc[geo_nodes["node_id"].isin(["N01", "N26", "N32", "N33", "N34"]),
                    [
                        "node_id", "representative_city", "longitude", "latitude",
                        "aeqd_x_km", "aeqd_y_km",
                    ]])
print("是否车站测量坐标：", reference["station_coordinates"], "；是否改动铁路时间矩阵：", reference["rail_time_matrix_changed"])
print("地理参考保存目录：", REFERENCE_OUT)


城市参考点数： 34 ；坐标系： OGC:CRS84 (WGS84 longitude, latitude order)
来源提交日期： 2022-05-13T23:22:03Z ；固定版本： 789c9904087846cc3361302857aa2e76b0ae71ff
距离方法： haversine great-circle distances on sphere; not Euclidean longitude/latitude
投影口径： {'method': 'spherical azimuthal equidistant', 'central_longitude': 105.0, 'central_latitude': 35.0, 'unit': 'km', 'proj_equivalent': '+proj=aeqd +lon_0=105 +lat_0=35 +R=6371008.8'}


,node_id,representative_city,longitude,latitude,aeqd_x_km,aeqd_y_km
0,N01,北京,116.394201,39.901720,970.760708,602.860532
25,N26,拉萨,91.100010,29.645024,-1341.454946,-505.836134
31,N32,香港,114.183064,22.306927,951.512957,-1372.217866
32,N33,澳门,113.544375,22.189135,886.337266,-1390.586060
33,N34,台北,121.568333,25.035833,1671.708521,-979.961220


是否车站测量坐标： False ；是否改动铁路时间矩阵： False
地理参考保存目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\analysis\user_stage2_geographic_reference_20261114_165102_851358


## 问题2—核心代码8：将固定代表嵌入与真实地理参考对齐

In [10]:
# 功能：读取已保存的8个代表模型，完成12组相似变换，并建立528个连通城市对的时间—地理表。
import stage2_geography
importlib.reload(stage2_geography)
geography = stage2_geography.analyze(actor="user")
for key in ["alignment_count", "representative_models", "source_city_count", "connected33_unique_pairs",
            "time_geography_spearman_33", "time_geography_spearman_32", "candidate_examples", "geographic_alignment_computed", "output_dir"]:
    print(f"{key}: {geography[key]}")


alignment_count: 12
representative_models: 8
source_city_count: 34
connected33_unique_pairs: 528
time_geography_spearman_33: 0.8742376453739596
time_geography_spearman_32: 0.88107652120738
candidate_examples: 20
geographic_alignment_computed: True
output_dir: D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\analysis\user_stage2_geographic_analysis_20261114_165132_458783


## 问题2—核心代码9：复核地理对齐并展示候选城市对

In [11]:
# 功能：核验来源、坐标、变换与全部地理指标；展示12行质量表和少量候选城市对。
import stage2_geography
importlib.reload(stage2_geography)
verified = stage2_geography.verify_results()
GEOGRAPHY_OUT = Path(verified["output_dir"])
alignment = pd.read_csv(GEOGRAPHY_OUT / "geographic_alignment_quality.csv", encoding="utf-8-sig")
examples = pd.read_csv(GEOGRAPHY_OUT / "candidate_city_pair_examples.csv", encoding="utf-8-sig")
with pd.option_context("display.max_columns", None, "display.width", 200, "display.precision", 5):
    display(alignment[[
        "dataset_id", "model", "fit_scope", "fitting_nodes",
        "projected_procrustes_disparity", "projected_rms_residual_km",
        "geo_distance_spearman",
    ]])
    display(examples.groupby("example_group", sort=False).head(5)[[
        "example_group", "city_a", "city_b", "geographic_great_circle_km",
        "symmetric_time_min", "time_minus_geographic_rank",
    ]])
print("地理对齐核验：", verified)


,dataset_id,model,fit_scope,fitting_nodes,projected_procrustes_disparity,projected_rms_residual_km,geo_distance_spearman
0,access_60_alpha_5_n34,MDS,all34,34,0.94606,1066.34456,0.78180
1,access_60_alpha_5_n34,MDS,network33,33,0.45861,738.61272,0.85871
2,access_60_alpha_5_n34,MDS_nonmetric,all34,34,0.95990,1074.11601,0.55699
3,access_60_alpha_5_n34,MDS_nonmetric,network33,33,0.91577,1043.73356,0.60210
4,access_60_alpha_5_n34,tSNE,all34,34,0.79440,977.14384,0.65362
5,access_60_alpha_5_n34,tSNE,network33,33,0.37787,670.44662,0.71568
6,access_60_alpha_5_n34,UMAP,all34,34,0.22738,522.77251,0.73580
7,access_60_alpha_5_n34,UMAP,network33,33,0.16710,445.85172,0.79397
8,access_60_n33,MDS,all33,33,0.43909,722.72314,0.86155
9,access_60_n33,MDS_nonmetric,all33,33,0.33251,628.92406,0.89192


,example_group,city_a,city_b,geographic_great_circle_km,symmetric_time_min,time_minus_geographic_rank
0,relatively_high_time_rank,海口,澳门,410.37984,2047.0,0.84848
1,relatively_high_time_rank,海口,香港,472.62428,2023.0,0.81250
2,relatively_high_time_rank,南宁,海口,371.38164,1299.5,0.76894
3,relatively_high_time_rank,海口,昆明,959.32189,1966.5,0.59091
4,relatively_high_time_rank,广州,海口,457.57220,703.0,0.58333
10,relatively_low_time_rank,北京,广州,1888.64612,437.0,-0.36458
11,relatively_low_time_rank,石家庄,广州,1663.86583,374.5,-0.36364
12,relatively_low_time_rank,北京,香港,1967.56153,491.5,-0.32860
13,relatively_low_time_rank,石家庄,香港,1750.99686,429.0,-0.32386
14,relatively_low_time_rank,北京,长沙,1338.39446,310.5,-0.27841


地理对齐核验： {'output_dir': 'D:\\yangz\\大三上\\机器学习\\第二次作业\\railway_analysis\\data\\analysis\\user_stage2_geographic_analysis_20261114_165132_458783', 'verified_alignments': 12, 'verified_coordinate_rows': 404, 'hashes_passed': True, 'similarity_transforms_reproduced': True, 'geographic_metrics_reproduced': True, 'connected33_unique_pairs': 528, 'geographic_comparison_computed_and_verified': True, 'external_economic_explanation_complete': False, 'clustering_complete': False}
